<a href="https://colab.research.google.com/github/Dhlih/Submission-Dicoding-Analisis-Sentimen/blob/main/Pelatihan_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Import Library

In [16]:
import pandas as pd  # Pandas untuk manipulasi dan analisis data
pd.options.mode.chained_assignment = None  # Menonaktifkan peringatan chaining

import numpy as np  # NumPy untuk komputasi numerik
import matplotlib.pyplot as plt  # Matplotlib untuk visualisasi data
import seaborn as sns  # Seaborn untuk visualisasi data statistik, mengatur gaya visualisasi
from sklearn.metrics import accuracy_score

import datetime as dt  # Manipulasi data waktu dan tanggal
import re  # Modul untuk bekerja dengan ekspresi reguler
import string  # Berisi konstanta string, seperti tanda baca

from nltk.tokenize import word_tokenize  # Tokenisasi teks
from nltk.corpus import stopwords  # Daftar kata-kata berhenti dalam teks

!pip install sastrawi
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory  # Stemming (penghilangan imbuhan kata) dalam bahasa Indonesia
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory  # Menghapus kata-kata berhenti dalam bahasa Indonesia

from wordcloud import WordCloud  # Membuat visualisasi berbentuk awan kata (word cloud) dari teks

import nltk  # Import pustaka NLTK (Natural Language Toolkit).
nltk.download('punkt_tab')  # Mengunduh dataset yang diperlukan untuk tokenisasi teks.
nltk.download('stopwords')  # Mengunduh dataset yang berisi daftar kata-kata berhenti (stopwords) dalam berbagai bahasa.

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## Loading Dataset

In [17]:
df = pd.read_csv("ulasan_ipusnas.csv")
df.head()

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,82e83362-0763-4eab-9e3d-fc11930634a8,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"gabisa minjem, kesalahan pengiriman data mulu,...",1,124,2.1.0,2026-02-25 12:21:00,NaN,NaN,2.1.0
1,eb1e116f-41d2-4699-bee6-a86918bbd023,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"kadang bisa minjem buku kadang engga, awalnya ...",2,234,2.1.0,2026-02-01 08:49:07,NaN,NaN,2.1.0
2,c28895ee-210a-4be9-81a3-7ad9c1a8b191,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"Ini kenapa setelah update data peminjaman, ant...",2,369,2.0.6,2026-01-26 15:43:13,NaN,NaN,2.0.6
3,d6273ac9-49fc-4d54-a80f-d01210795215,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Halo ipusnas! Kenapa dari bulan Januari sampai...,3,31,2.1.0,2026-02-23 06:17:42,NaN,NaN,2.1.0
4,682da2e3-3176-4589-87bb-956495cf64a7,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Aplikasi ini tujuannya memudahkan akses baca s...,1,185,2.1.0,2026-02-22 07:15:35,NaN,NaN,2.1.0


In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19959 entries, 0 to 19958
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   reviewId              19959 non-null  object
 1   userName              19959 non-null  object
 2   userImage             19959 non-null  object
 3   content               19959 non-null  object
 4   score                 19959 non-null  int64 
 5   thumbsUpCount         19959 non-null  int64 
 6   reviewCreatedVersion  14981 non-null  object
 7   at                    19959 non-null  object
 8   replyContent          13130 non-null  object
 9   repliedAt             13130 non-null  object
 10  appVersion            14981 non-null  object
dtypes: int64(2), object(9)
memory usage: 1.7+ MB


In [19]:
df.isnull().sum()

,0
reviewId,0
userName,0
userImage,0
content,0
score,0
thumbsUpCount,0
reviewCreatedVersion,4978
at,0
replyContent,6829
repliedAt,6829


In [20]:
clean_df = df.dropna()
clean_df.isnull().sum()

,0
reviewId,0
userName,0
userImage,0
content,0
score,0
thumbsUpCount,0
reviewCreatedVersion,0
at,0
replyContent,0
repliedAt,0


In [21]:
clean_df.duplicated().sum()

np.int64(0)

## Preprocess Text

In [30]:
import re
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

def cleaningText(text):
    text = re.sub(r'@[A-Za-z0-9]+', '', text) # menghapus mention
    text = re.sub(r'#[A-Za-z0-9]+', '', text) # menghapus hashtag
    text = re.sub(r'RT[\s]', '', text) # menghapus RT
    text = re.sub(r"http\S+", '', text) # menghapus link
    text = re.sub(r'[0-9]+', '', text) # menghapus angka
    text = re.sub(r'[^\w\s]', '', text) # menghapus karakter selain huruf dan angka

    text = text.replace('\n', ' ') # mengganti baris baru dengan spasi
    text = text.translate(str.maketrans('', '', string.punctuation)) # menghapus semua tanda baca
    text = text.strip(' ') # menghapus karakter spasi dari kiri dan kanan teks
    return text

def casefoldingText(text): # Mengubah semua karakter dalam teks menjadi huruf kecil
    text = text.lower()
    return text

def tokenizingText(text): # Memecah atau membagi string, teks menjadi daftar token
    text = word_tokenize(text)
    return text

def toSentence(list_words): # Mengubah daftar kata menjadi kalimat
    sentence = ' '.join(word for word in list_words)
    return sentence

In [60]:
slang_words

{'@': 'di',
 'abis': 'habis',
 'ad': 'ada',
 'adlh': 'adalah',
 'afaik': 'as far as i know',
 'ahaha': 'haha',
 'aj': 'saja',
 'ajep-ajep': 'dunia gemerlap',
 'ak': 'saya',
 'akika': 'aku',
 'akkoh': 'aku',
 'akuwh': 'aku',
 'alay': 'norak',
 'alow': 'halo',
 'ambilin': 'ambilkan',
 'ancur': 'hancur',
 'anjrit': 'anjing',
 'anter': 'antar',
 'ap2': 'apa-apa',
 'apasih': 'apa sih',
 'apes': 'sial',
 'aps': 'apa',
 'aq': 'saya',
 'aquwh': 'aku',
 'asbun': 'asal bunyi',
 'aseekk': 'asyik',
 'asekk': 'asyik',
 'asem': 'asam',
 'aspal': 'asli tetapi palsu',
 'astul': 'asal tulis',
 'ato': 'atau',
 'au ah': 'tidak mau tahu',
 'awak': 'saya',
 'ay': 'sayang',
 'ayank': 'sayang',
 'b4': 'sebelum',
 'bakalan': 'akan',
 'bandes': 'bantuan desa',
 'bangedh': 'banget',
 'banpol': 'bantuan polisi',
 'banpur': 'bantuan tempur',
 'basbang': 'basi',
 'bcanda': 'bercanda',
 'bdg': 'bandung',
 'begajulan': 'nakal',
 'beliin': 'belikan',
 'bencong': 'banci',
 'bentar': 'sebentar',
 'ber3': 'bertiga',
 'b

In [78]:
import json

with open('slang_words.json', 'r') as f:
    slang_words = json.load(f)

def fix_slangwords(text):
    words = text.split()
    print(words)
    fixed_words = []

    for word in words:
        if word.lower() in slang_words:
            fixed_words.append(slang_words[word.lower()])
        else:
            fixed_words.append(word)

    fixed_text = ' '.join(fixed_words)
    return fixed_text

with open('stop_words.txt', 'r') as f:
    stop_words = set(f.read().splitlines())

def filteringText(text): # Menghapus stopwords dalam teks
    filtered = []
    for txt in text:
        if txt not in stop_words:
            filtered.append(txt)
    text = filtered
    return text

In [31]:
# Membersihkan teks dan menyimpannya di kolom 'text_clean'
clean_df['text_clean'] = clean_df['content'].apply(cleaningText)

# Mengubah huruf dalam teks menjadi huruf kecil dan menyimpannya di 'text_casefoldingText'
clean_df['text_casefoldingText'] = clean_df['text_clean'].apply(casefoldingText)

# Mengganti kata-kata slang dengan kata-kata standar dan menyimpannya di 'text_slangwords'
clean_df['text_slangwords'] = clean_df['text_casefoldingText'].apply(fix_slangwords)

# Memecah teks menjadi token (kata-kata) dan menyimpannya di 'text_tokenizingText'
clean_df['text_tokenizingText'] = clean_df['text_slangwords'].apply(tokenizingText)

# Menghapus kata-kata stop (kata-kata umum) dan menyimpannya di 'text_stopword'
clean_df['text_stopword'] = clean_df['text_tokenizingText'].apply(filteringText)

# Menggabungkan token-token menjadi kalimat dan menyimpannya di 'text_akhir'
clean_df['text_akhir'] = clean_df['text_stopword'].apply(toSentence)

In [33]:
clean_df.head()

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,text_clean,text_casefoldingText,text_slangwords,text_tokenizingText,text_stopword,text_akhir
23,d5f0a704-9260-47f4-960d-153ab64d6cf4,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"Setelah diupdate krn keperluan tugas, rupanya ...",1,46,2.0.6,2026-01-29 01:38:27,Hallo Liliy! Terima kasih sudah memberikan ula...,2022-03-27 08:52:58,2.0.6,Setelah diupdate krn keperluan tugas rupanya g...,setelah diupdate krn keperluan tugas rupanya g...,setelah diupdate krn keperluan tugas rupanya g...,"[setelah, diupdate, krn, keperluan, tugas, rup...","[setelah, diupdate, krn, keperluan, tugas, rup...",setelah diupdate krn keperluan tugas rupanya g...
28,27609d0c-739b-4a34-847b-ff4a6a8c3fdc,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Saya Download Aplikasi Ipusnas Langsung di Pla...,1,12,2.0.6,2026-01-25 02:42:18,Hallo Usman! Kami mohon maaf atas kendala dan ...,2023-04-01 11:01:42,2.0.6,Saya Download Aplikasi Ipusnas Langsung di Pla...,saya download aplikasi ipusnas langsung di pla...,saya download aplikasi ipusnas langsung di pla...,"[saya, download, aplikasi, ipusnas, langsung, ...","[saya, download, aplikasi, ipusnas, langsung, ...",saya download aplikasi ipusnas langsung di pla...
33,96709d16-28e3-4dc9-aef2-7562a85b129e,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Terima kasih sudah memberi kesempatan untuk ya...,2,3,2.0.6,2026-01-24 23:28:33,Hallo Puristi! Kami mohon maaf atas ketidaknya...,2024-09-09 01:34:17,2.0.6,Terima kasih sudah memberi kesempatan untuk ya...,terima kasih sudah memberi kesempatan untuk ya...,terima kasih sudah memberi kesempatan untuk ya...,"[terima, kasih, sudah, memberi, kesempatan, un...","[terima, kasih, sudah, memberi, kesempatan, un...",terima kasih sudah memberi kesempatan untuk ya...
41,03dc5e28-0e6f-403a-87f1-694e00c29b5d,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,astaga ipusnas... kenapa makin jelek aja sih.....,1,15,2.0.6,2026-01-29 23:28:07,Hallo Pujiana! Kami mohon maaf atas ketidaknya...,2023-10-01 07:16:16,2.0.6,astaga ipusnas kenapa makin jelek aja sih sebe...,astaga ipusnas kenapa makin jelek aja sih sebe...,astaga ipusnas kenapa makin jelek aja sih sebe...,"[astaga, ipusnas, kenapa, makin, jelek, aja, s...","[astaga, ipusnas, kenapa, makin, jelek, aja, s...",astaga ipusnas kenapa makin jelek aja sih sebe...
54,b74d631a-0828-44db-bea5-8d459c8ca039,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Koleksi bukunya banyak banget! Meski harus nga...,5,2,2.1.0,2026-02-13 03:10:28,Hallo Pine! Kami mohon maaf atas kendala dan k...,2024-05-20 22:44:26,2.1.0,Koleksi bukunya banyak banget Meski harus ngan...,koleksi bukunya banyak banget meski harus ngan...,koleksi bukunya banyak banget meski harus ngan...,"[koleksi, bukunya, banyak, banget, meski, haru...","[koleksi, bukunya, banyak, banget, meski, haru...",koleksi bukunya banyak banget meski harus ngan...
